# Do the singleton recommendations make sense?

## Executive conclusion

**The displayed recommendations for *The Lord of the Rings* do not provide convincing LOTR-specific recommendations.** They are defensible as books read by overlapping users, but the evidence is extremely sparse and dominated by globally popular books.

Using the training split that produced the model mappings:

| Recommendation | Global popularity rank | Readers shared with the 13 LOTR readers | Co-reader lift |
|---|---:|---:|---:|
| Wild Animus | 1 | 2 | 7.57 |
| The Lovely Bones | 2 | 2 | 14.31 |
| The Da Vinci Code | 3 | 2 | 21.63 |
| Snow Falling on Cedars | 8 | 1 | 15.82 |
| Angels & Demons | 10 | 2 | 32.79 |

Lift above one means the pairs co-occur more often than independence would predict, so the output is not purely arbitrary. However, one or two shared readers is too little support for a strong semantic claim, and all five books being top-10 items is clear popularity dominance. None is an obvious Tolkien, epic-fantasy, or closely related recommendation. The sigmoid values near `0.99` are sampled-BCE ranking scores, not calibrated probabilities that a reader will like a book.

The remaining cells reproduce this audit, inspect several unrelated singleton queries, and compare singleton test ranking against Most Popular before producing a broader model-level verdict.

In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

from bookrec.data import (
    ITEM_COLUMN,
    USER_COLUMN,
    encode_interactions,
    load_dataset,
    split_interactions,
)
from bookrec.implicit.baselines import MostPopularBaseline
from bookrec.implicit.datasets import (
    SampledRankingDataset,
    collate_implicit_batch,
)
from bookrec.implicit.evaluation import evaluate_sampled_ranking
from bookrec.implicit.hyperparameters import DEFAULT_HYPERPARAMETERS
from bookrec.implicit.model import HISTORY_MLP_ARCHITECTURE_VERSION, create_implicit_model

ARTIFACTS = Path("artifacts")
CHECKPOINT_PATH = ARTIFACTS / "implicit" / "history_mlp" / "model_with_mappings.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_SEED = 42
TOP_K = 5
SCORING_BATCH_SIZE = 16_384
EVALUATION_BATCH_SIZE = 64
EVALUATION_CANDIDATES = 1_000
MIN_RECOMMENDATION_INTERACTIONS = 5

LOTR_ISBN = "0618260250"
OBSERVED_RECOMMENDATION_ISBNS = [
    "0971880107",  # Wild Animus
    "0316666343",  # The Lovely Bones
    "0671027360",  # Angels & Demons
    "067976402X",  # Snow Falling on Cedars
    "0385504209",  # The Da Vinci Code
]
QUERY_TITLES = [
    "The Lord of the Rings",
    "Julius Caesar (Oxford School Shakespeare)",
    "The Da Vinci Code",
    "The Notebook",
    "Harry Potter and the Sorcerer's Stone",
    "Pride and Prejudice",
]

print(f"Device: {DEVICE}")

## 1. Reproduce the training-data audit

Only the training split is used for popularity and co-reader statistics. This prevents validation and test interactions from leaking into the explanation.

In [ ]:
ratings = load_dataset("Ratings.csv")
books = load_dataset("Books.csv")
ratings[ITEM_COLUMN] = ratings[ITEM_COLUMN].astype(str)
books[ITEM_COLUMN] = books[ITEM_COLUMN].astype(str)

metadata = (
    books[[ITEM_COLUMN, "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]]
    .drop_duplicates(ITEM_COLUMN)
)
train_raw, validation_raw, test_raw = split_interactions(
    ratings, seed=SPLIT_SEED
)
train_counts = train_raw[ITEM_COLUMN].value_counts()
popularity_rank = train_counts.rank(method="min", ascending=False)
num_train_users = train_raw[USER_COLUMN].nunique()

audited_isbns = [LOTR_ISBN, *OBSERVED_RECOMMENDATION_ISBNS]
reader_sets = (
    train_raw[train_raw[ITEM_COLUMN].isin(audited_isbns)]
    .groupby(ITEM_COLUMN)[USER_COLUMN]
    .agg(set)
    .to_dict()
)
lotr_readers = reader_sets.get(LOTR_ISBN, set())

audit_rows = []
for isbn in OBSERVED_RECOMMENDATION_ISBNS:
    candidate_readers = reader_sets.get(isbn, set())
    common_readers = len(lotr_readers & candidate_readers)
    denominator = len(lotr_readers) * len(candidate_readers)
    lift = (
        common_readers * num_train_users / denominator
        if denominator
        else np.nan
    )
    audit_rows.append({
        ITEM_COLUMN: isbn,
        "LOTR readers": len(lotr_readers),
        "candidate readers": len(candidate_readers),
        "shared readers": common_readers,
        "co-reader lift": lift,
        "global popularity rank": int(popularity_rank.get(isbn, 0)),
    })

observed_audit = (
    pd.DataFrame(audit_rows)
    .merge(metadata, on=ITEM_COLUMN, how="left")
    [[
        "Book-Title",
        "Book-Author",
        ITEM_COLUMN,
        "global popularity rank",
        "LOTR readers",
        "candidate readers",
        "shared readers",
        "co-reader lift",
    ]]
    .sort_values("global popularity rank")
)
display(observed_audit.style.format({"co-reader lift": "{:.2f}"}))

### Interpretation

The positive lift values show a real collaborative signal: these books occur among LOTR readers more often than their base rates predict. But lift is unstable when the query has only 13 readers, and every estimate rests on only one or two shared people. The candidates' popularity ranks of 1, 2, 3, 8, and 10 are much stronger evidence that the top of the ranking is popularity-dominated.

This dataset contains no genre field, so semantic relevance cannot be measured automatically here. A manual title-level check finds no obvious epic-fantasy or Tolkien connection in the displayed top five.

## 2. Load the model and inspect several singleton queries

A useful singleton model should change its ranking when the input book changes. The next cells resolve several contrasting titles, including Julius Caesar, score the mapped catalog, discard candidates with fewer than five training interactions, and compare their recommendation sets.

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
if checkpoint.get("model_type") != "history_mlp":
    raise ValueError(f"{CHECKPOINT_PATH} is not a history_mlp checkpoint")
if checkpoint.get("architecture_version") != HISTORY_MLP_ARCHITECTURE_VERSION:
    raise ValueError("The history_mlp checkpoint is obsolete; retrain it first")

user_to_index = checkpoint["user_to_index"]
item_to_index = checkpoint["item_to_index"]
hyperparameters = checkpoint.get("hyperparameters", DEFAULT_HYPERPARAMETERS)
model = create_implicit_model(
    "history_mlp",
    num_users=len(user_to_index),
    num_items=len(item_to_index),
    hyperparameters=hyperparameters,
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

index_to_isbn = [None] * len(item_to_index)
for isbn, item_index in item_to_index.items():
    index_to_isbn[item_index] = str(isbn)
mapped_metadata = metadata[metadata[ITEM_COLUMN].isin(item_to_index)].copy()
mapped_metadata_isbns = set(mapped_metadata[ITEM_COLUMN])


def resolve_title(title):
    titles = mapped_metadata["Book-Title"].astype(str)
    exact = titles.str.casefold() == title.casefold()
    matches = mapped_metadata[exact].copy()
    if matches.empty:
        contains = titles.str.contains(title, case=False, regex=False, na=False)
        matches = mapped_metadata[contains].copy()
    if matches.empty:
        return None
    matches["train interactions"] = (
        matches[ITEM_COLUMN].map(train_counts).fillna(0)
    )
    return matches.nlargest(1, "train interactions").iloc[0]


def score_singleton(query_isbn):
    history_items = torch.tensor(
        [item_to_index[query_isbn]], dtype=torch.long, device=DEVICE
    )
    history_offset = torch.tensor([0], dtype=torch.long, device=DEVICE)
    logits = []
    with torch.inference_mode():
        for start in range(0, len(index_to_isbn), SCORING_BATCH_SIZE):
            stop = min(start + SCORING_BATCH_SIZE, len(index_to_isbn))
            items = torch.arange(start, stop, device=DEVICE).unsqueeze(0)
            users = torch.zeros_like(items)  # history_mlp ignores users.
            batch_logits = model(
                users=users,
                items=items,
                history_items=history_items,
                history_offset=history_offset,
            )
            logits.append(batch_logits.squeeze(0).cpu())
    return pd.DataFrame({
        ITEM_COLUMN: index_to_isbn,
        "logit": torch.cat(logits).numpy(),
    })


def recommend_singleton(query_title, top_k=TOP_K):
    query = resolve_title(query_title)
    if query is None:
        print(f"Skipping unmapped title: {query_title!r}")
        return None, None
    query_isbn = query[ITEM_COLUMN]
    scores = score_singleton(query_isbn)
    scores = scores[
        scores[ITEM_COLUMN].isin(mapped_metadata_isbns)
        & scores[ITEM_COLUMN].isin(
            train_counts[
                train_counts >= MIN_RECOMMENDATION_INTERACTIONS
            ].index
        )
        & (scores[ITEM_COLUMN] != query_isbn)
    ]
    recommendations = (
        scores.nlargest(top_k, "logit")
        .merge(metadata, on=ITEM_COLUMN, how="left")
    )
    recommendations["sigmoid score"] = torch.sigmoid(
        torch.tensor(recommendations["logit"].to_numpy())
    ).numpy()
    return query, recommendations


query_rows = []
recommendation_frames = []
for requested_title in QUERY_TITLES:
    query, recommendations = recommend_singleton(requested_title)
    if query is None:
        continue
    query_rows.append({
        "requested title": requested_title,
        "resolved title": query["Book-Title"],
        "query ISBN": query[ITEM_COLUMN],
        "query train interactions": int(train_counts.get(query[ITEM_COLUMN], 0)),
    })
    recommendations = recommendations.copy()
    recommendations["query title"] = query["Book-Title"]
    recommendations["query ISBN"] = query[ITEM_COLUMN]
    recommendations["rank"] = np.arange(1, len(recommendations) + 1)
    recommendation_frames.append(recommendations)

resolved_queries = pd.DataFrame(query_rows)
all_recommendations = pd.concat(recommendation_frames, ignore_index=True)
display(resolved_queries)
display(all_recommendations[[
    "query title", "rank", "Book-Title", "Book-Author", ITEM_COLUMN, "sigmoid score"
]])

## 3. Quantify popularity dominance and query sensitivity

Three diagnostics are used:

- **Popularity overlap@5:** fraction of a query's recommendations also found in the five most-interacted mapped books. Lower is better for personalization.
- **Pairwise recommendation Jaccard:** overlap between top-five lists for unrelated inputs. Lower means the history changes the output.
- **Co-reader support and lift:** support counts how many training users read both books; lift compares that count with independence. Lift without support is fragile.

In [ ]:
mapped_popularity = train_counts[train_counts.index.isin(mapped_metadata_isbns)]
recommendation_sets = {
    query_isbn: set(group[ITEM_COLUMN])
    for query_isbn, group in all_recommendations.groupby("query ISBN")
}
popularity_rows = []
for query_isbn, recommended in recommendation_sets.items():
    query_title = all_recommendations.loc[
        all_recommendations["query ISBN"] == query_isbn, "query title"
    ].iloc[0]
    popular_for_query = set(
        mapped_popularity.drop(index=query_isbn, errors="ignore")
        .nlargest(TOP_K)
        .index
    )
    popularity_rows.append({
        "query title": query_title,
        "popularity overlap@5": len(recommended & popular_for_query) / TOP_K,
        "median recommendation popularity rank": float(
            popularity_rank.reindex(list(recommended)).median()
        ),
    })
popularity_diagnostics = pd.DataFrame(popularity_rows)

jaccard_rows = []
for first_isbn, second_isbn in combinations(recommendation_sets, 2):
    first = recommendation_sets[first_isbn]
    second = recommendation_sets[second_isbn]
    jaccard_rows.append({
        "first query": first_isbn,
        "second query": second_isbn,
        "top-5 Jaccard": len(first & second) / len(first | second),
    })
jaccard_diagnostics = pd.DataFrame(jaccard_rows)

needed_isbns = set(recommendation_sets) | set(all_recommendations[ITEM_COLUMN])
needed_reader_sets = (
    train_raw[train_raw[ITEM_COLUMN].isin(needed_isbns)]
    .groupby(ITEM_COLUMN)[USER_COLUMN]
    .agg(set)
    .to_dict()
)
association_rows = []
for _, row in all_recommendations.iterrows():
    query_isbn = row["query ISBN"]
    candidate_isbn = row[ITEM_COLUMN]
    query_readers = needed_reader_sets.get(query_isbn, set())
    candidate_readers = needed_reader_sets.get(candidate_isbn, set())
    common = len(query_readers & candidate_readers)
    denominator = len(query_readers) * len(candidate_readers)
    association_rows.append({
        "query ISBN": query_isbn,
        ITEM_COLUMN: candidate_isbn,
        "shared readers": common,
        "co-reader lift": (
            common * num_train_users / denominator if denominator else np.nan
        ),
    })
association_diagnostics = pd.DataFrame(association_rows)

display(popularity_diagnostics)
display(jaccard_diagnostics)
display(association_diagnostics.describe())

## 4. Offline singleton ranking against Most Popular

The final objective check uses the real test protocol: one deterministic known book per user is the context, held-out test interactions are relevant candidates, and both models rank identical sampled candidates. This takes a few minutes on the full dataset.

In [ ]:
train_data = encode_interactions(train_raw, user_to_index, item_to_index)
validation_data = encode_interactions(
    validation_raw, user_to_index, item_to_index
)
test_data = encode_interactions(test_raw, user_to_index, item_to_index)
test_context = pd.concat([train_data, validation_data], ignore_index=True)
all_interactions = pd.concat(
    [train_data, validation_data, test_data], ignore_index=True
)
singleton_test = SampledRankingDataset(
    held_out_interactions=test_data,
    context_interactions=test_context,
    all_interactions=all_interactions,
    num_items=len(item_to_index),
    num_candidates=EVALUATION_CANDIDATES,
    seed=SPLIT_SEED + 1,
    context_mode="singleton",
)
singleton_loader = DataLoader(
    singleton_test,
    batch_size=EVALUATION_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_implicit_batch,
)
popularity_model = MostPopularBaseline.fit(
    train_data, num_items=len(item_to_index)
).to(DEVICE)

history_scores = evaluate_sampled_ranking(model, singleton_loader, DEVICE)
popularity_scores = evaluate_sampled_ranking(
    popularity_model, singleton_loader, DEVICE
)
offline_results = pd.DataFrame([
    {"model": "history_mlp singleton", **history_scores},
    {"model": "most popular", **popularity_scores},
]).set_index("model")
display(offline_results)

## 5. Evidence-based model verdict

The rubric below intentionally requires more than attractive titles or large sigmoid scores. A singleton recommender should beat popularity on held-out ranking, respond to the query, avoid simply returning the global top five, and have more than one or two co-readers supporting a typical recommendation. Thresholds are explicit so they can be challenged rather than hidden in a subjective conclusion.

In [ ]:
mean_popularity_overlap = popularity_diagnostics["popularity overlap@5"].mean()
mean_pairwise_jaccard = (
    jaccard_diagnostics["top-5 Jaccard"].mean()
    if not jaccard_diagnostics.empty
    else 1.0
)
median_shared_readers = association_diagnostics["shared readers"].median()
history_ndcg = history_scores["ndcg_at_50"]
popularity_ndcg = popularity_scores["ndcg_at_50"]

checks = pd.DataFrame([
    {
        "criterion": "Beats Most Popular on singleton NDCG@50",
        "observed": history_ndcg - popularity_ndcg,
        "passes": history_ndcg > popularity_ndcg,
    },
    {
        "criterion": "Mean popularity overlap@5 below 40%",
        "observed": mean_popularity_overlap,
        "passes": mean_popularity_overlap < 0.40,
    },
    {
        "criterion": "Mean cross-query Jaccard below 30%",
        "observed": mean_pairwise_jaccard,
        "passes": mean_pairwise_jaccard < 0.30,
    },
    {
        "criterion": "Median co-reader support at least 3",
        "observed": median_shared_readers,
        "passes": median_shared_readers >= 3,
    },
])
display(checks)

passed = int(checks["passes"].sum())
if passed == len(checks):
    verdict = (
        "YES: the broader evidence supports meaningful singleton recommendations. "
        "The displayed LOTR list remains a weak sparse-query example."
    )
elif passed >= 2:
    verdict = (
        "PARTIALLY: the model uses some query information, but the singleton "
        "recommendations are not consistently stronger than popularity."
    )
else:
    verdict = (
        "NO: the singleton recommendations are primarily popularity-driven or "
        "too weakly supported to claim that they reflect the input book."
    )

print(f"Passed {passed}/{len(checks)} checks")
print(verdict)